In [1]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import joblib

# -----------------------------
# 1. Create fake training data
# -----------------------------

np.random.seed(42)

X = np.random.uniform(
    low=[20, -0.005],
    high=[80, 0.005],
    size=(1000, 2)
).astype(np.float32)

rsi = X[:, 0]
ema_diff = X[:, 1]

# Simple artificial rule:
# BUY if RSI > 50 and EMA difference > 0
y = ((rsi > 50) & (ema_diff > 0)).astype(np.int64)


# -----------------------------
# 2. Train model
# -----------------------------

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

print("Model trained!")


# -----------------------------
# 3. Test Python model
# -----------------------------

test_data = np.array([
    [65, 0.002],
    [35, -0.002],
    [70, -0.001],
    [45, 0.003]
], dtype=np.float32)

predictions = model.predict(test_data)

print("Python predictions:")
print(predictions)


# -----------------------------
# 4. Export to ONNX
# -----------------------------

initial_type = [
    (
        "float_input",
        FloatTensorType([None, 2])
    )
]

onnx_model = convert_sklearn(
    model,
    initial_types=initial_type
)

with open("trading_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("ONNX model saved as trading_model.onnx")


Model trained!
Python predictions:
[1 0 0 0]
ONNX model saved as trading_model.onnx


In [2]:
import numpy as np
import onnxruntime as ort

# Load ONNX model
session = ort.InferenceSession("trading_model.onnx")

# Check input name
input_name = session.get_inputs()[0].name

print("Input name:", input_name)

# New data
X = np.array([
    [65, 0.002],
    [35, -0.002]
], dtype=np.float32)

# Run model
result = session.run(
    None,
    {input_name: X}
)

print("ONNX output:")
print(result)


Input name: float_input
ONNX output:
[array([1, 0]), [{0: 6.556510925292969e-07, 1: 0.9999993443489075}, {0: 1.0, 1: 0.0}]]
